# Threshold pseudo-labeling

This method trains a neural network on labels produced by `detect_broadband_bursts.py` logic.

Workflow:

1. Run `prepare_threshold_dataset.py` to build `dataset/` inside this method.
2. Train the 1D CNN on the pseudo-labeled `envelope_db` arrays.
3. Put manually verified PNGs into `dataset/verify/burst` and `dataset/verify/no_burst` for an independent check.

## Prepare dataset

Run this from PowerShell in the project root if the method dataset is empty:

```powershell
python .\methods\threshold_pseudo_labeling\prepare_threshold_dataset.py .\raw_data --channel ns --freq-min 20000 --freq-max 30000 --time-resolution 0.002 --frequency-resolution 50 --segment-duration 2 --threshold-mad 8 --overwrite
```

The script creates:

```text
methods/threshold_pseudo_labeling/dataset/samples
methods/threshold_pseudo_labeling/dataset/review/burst
methods/threshold_pseudo_labeling/dataset/review/no_burst
methods/threshold_pseudo_labeling/dataset/verify/burst
methods/threshold_pseudo_labeling/dataset/verify/no_burst
methods/threshold_pseudo_labeling/dataset/metadata.csv
```

In [ ]:
from pathlib import Path
import sys

METHOD_DIR = Path.cwd()
if METHOD_DIR.name != "threshold_pseudo_labeling":
    METHOD_DIR = Path("methods/threshold_pseudo_labeling").resolve()
sys.path.insert(0, str(METHOD_DIR))

from train_pseudo_envelope_1d_cnn import INDEX_TO_LABEL, LABEL_TO_INDEX, train

DATASET_DIR = METHOD_DIR / "dataset"
DATASET_DIR


In [ ]:
import pandas as pd

metadata_path = DATASET_DIR / "metadata.csv"
if not metadata_path.exists():
    raise FileNotFoundError("Run prepare_threshold_dataset.py first; metadata.csv is missing.")

metadata = pd.read_csv(metadata_path)
print(f"samples: {len(metadata)}")
display(metadata.groupby(["split", "label"]).size().unstack(fill_value=0))
metadata.head()


In [ ]:
EPOCHS = 40
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 42

model, report, predictions = train(
    dataset_dir=DATASET_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    output_dir=METHOD_DIR,
)


## Manual sample check

Set `SAMPLE` to a pseudo sample id, `.npz` path, or `.png` path from this method dataset.

In [ ]:
import numpy as np
import torch
from IPython.display import Image, display

metadata = pd.read_csv(DATASET_DIR / "metadata.csv")
metadata = metadata[metadata["label"].isin(LABEL_TO_INDEX)].copy()

def row_for_sample(sample: str):
    sample = str(sample).strip().strip('"')
    sample_path = Path(sample.replace("\\", "/"))
    sample_id = sample_path.stem if sample_path.suffix else sample
    rows = metadata[metadata["sample_id"] == sample_id]
    if rows.empty:
        rows = metadata[metadata["npz_path"].astype(str).str.replace("\\", "/", regex=False).str.endswith(sample_path.name)]
    if rows.empty:
        rows = metadata[metadata["png_path"].astype(str).str.replace("\\", "/", regex=False).str.endswith(sample_path.name)]
    if rows.empty:
        raise ValueError(f"Sample not found in metadata: {sample}")
    return rows.iloc[0]

def predict_sample(sample: str, show_image: bool = True):
    row = row_for_sample(sample)
    npz_path = DATASET_DIR / Path(str(row["npz_path"]).replace("\\", "/"))
    png_path = DATASET_DIR / Path(str(row["png_path"]).replace("\\", "/"))
    data = np.load(npz_path)
    x = data["envelope_db"].astype(np.float32)
    x = (x - x.mean()) / (x.std() + 1e-6)
    x = torch.from_numpy(x[None, None, :])
    model.eval()
    with torch.no_grad():
        prob = torch.softmax(model(x), dim=1).cpu().numpy()[0]
    predicted = INDEX_TO_LABEL[int(prob.argmax())]
    print(f"sample_id     : {row['sample_id']}")
    print(f"detector label: {row['label']}")
    print(f"predicted     : {predicted}")
    print(f"prob_no_burst : {prob[0]:.4f}")
    print(f"prob_burst    : {prob[1]:.4f}")
    print(f"peak_count    : {row.get('peak_count', '')}")
    if show_image and png_path.exists():
        display(Image(filename=str(png_path)))
    return {"sample_id": row["sample_id"], "label": row["label"], "predicted": predicted, "prob_burst": float(prob[1])}

SAMPLE = "pseudo_000000"
predict_sample(SAMPLE)


## Verify dataset check

Put manually checked PNG files into:

```text
dataset/verify/burst
dataset/verify/no_burst
```

Keep the original pseudo filename, for example `pseudo_000123.png`. The notebook finds the corresponding `.npz` through `metadata.csv`.

In [ ]:
VERIFY_DIR = DATASET_DIR / "verify"
(VERIFY_DIR / "burst").mkdir(parents=True, exist_ok=True)
(VERIFY_DIR / "no_burst").mkdir(parents=True, exist_ok=True)

def predict_verify_folder():
    rows = []
    for expected_label in ["no_burst", "burst"]:
        folder = VERIFY_DIR / expected_label
        for png_path in sorted(folder.glob("*.png")):
            result = predict_sample(png_path.name, show_image=False)
            result["expected_label"] = expected_label
            result["verify_png"] = str(png_path)
            result["correct"] = result["predicted"] == expected_label
            rows.append(result)
    if not rows:
        print(f"No PNG files found in {VERIFY_DIR / 'burst'} or {VERIFY_DIR / 'no_burst'}")
        return pd.DataFrame()
    verify = pd.DataFrame(rows)
    display(verify.groupby(["expected_label", "predicted"]).size().unstack(fill_value=0))
    print(f"accuracy: {verify['correct'].mean():.4f} ({verify['correct'].sum()}/{len(verify)})")
    return verify.sort_values("prob_burst", ascending=False)

verify_predictions = predict_verify_folder()
verify_predictions.head(20)
